# 03 - Feature engineeringPure transformation of `01_collected/` into `03_features/`. No network access - that is what makes this step reproducible.**Critical:** the n-gram model is fitted on TRAINING benign domains only. Fitting on the full corpus leaks the test distribution into the features.

In [ ]:
# --- standard header: every notebook starts with exactly this ---from google.colab import drive; drive.mount('/content/drive')REPO = '/content/secure-dns-trust-ai'!git -C {REPO} pull -q 2>/dev/null || git clone -q https://github.com/sandesh20lamichhane/secure-dns-trust-ai.git {REPO}import sys, os; sys.path.insert(0, REPO)os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'%load_ext autoreload%autoreload 2from src.utils import config, manifest, seeds, ioP = config.paths(); config.ensure_tree(P); seeds.set_all(42)print('repo', manifest.git_sha(REPO))

In [ ]:
import pandas as pdfrom src.utils.io import read_shardsfrom src.features import lexical, buildtls = read_shards(f"{P['data']['collected']}/tls_probe", 'tls_probe')ct  = read_shards(f"{P['data']['collected']}/crtsh", 'crtsh')dns = read_shards(f"{P['data']['collected']}/dns_records", 'dns')labels = pd.read_parquet(f"{P['data']['interim']}/domains_labelled.parquet")print(len(tls), len(ct), len(dns), len(labels))

In [ ]:
from src.evaluate.splits import load_splitsplit = load_split(P['data']['splits'], 'family_disjoint_v1')train_benign = labels[(labels['label']==0) & labels['domain'].isin(split['domains']['train'])]['domain']lexical.fit_ngram_model(train_benign)   # TRAIN ONLY

In [ ]:
cfg = config.load('features')groups = [g for g, v in cfg['groups'].items() if v.get('enabled') and g != 'embedding']X = build.assemble(labels, tls_df=tls, ct_df=ct, dns_df=dns,                   groups=groups, quarantined=cfg['quarantined'])X.to_parquet(f"{P['data']['features']}/fused_v1.parquet", index=False, compression='zstd')print(X.shape); X.head()